# Notebook 2 — Constraint Testing

Tests every constraint function using randomly generated schedules.
Each test builds data that is designed to trigger a known number of violations,
so every assertion can be reasoned about without looking anything up.

| Label | Function | Type |
|-------|----------|------|
| CT1 | `no_room_double_booking` | Hard |
| CT2a | `no_student_clash` | Hard |
| CT2b | `one_exam_per_student_per_day` | Soft |
| CT3 | `exams_spread_evenly` | Soft |
| CT4 | `assign_rooms_to_exam` + room assignment check | Hard |
| — | `hard_violations` | Aggregator |
| — | `soft_violations` | Aggregator |

---
## Step 0 — Imports

In [1]:
import sys
import os
import random
import importlib

sys.path.append(os.path.abspath('..'))

import utils.constraints
importlib.reload(utils.constraints)

from utils.constraints import (
    no_room_double_booking,
    no_student_clash,
    one_exam_per_student_per_day,
    exams_spread_evenly,
    assign_rooms_to_exam,
    hard_violations,
    soft_violations,
    explain_violations,
)

print('All imports successful.')

All imports successful.


---
## Step 1 — Random Test Data

All tests share this randomly generated dataset.
A fixed seed is used so results are reproducible every run.

In [2]:
random.seed(42)

# ── Students ──────────────────────────────────────────────────────────────────
# 30 students, each enrolled in 2–4 randomly chosen exams.
EXAM_CODES = ['CS101','CS102','CS201','MA101','MA102','MA201','PH101','PH102','EE101','EE102']
NUM_STUDENTS = 30

student_enrollments = {}
for i in range(NUM_STUDENTS):
    sid = f'S{i+1:03}'
    k   = random.randint(2, 4)
    student_enrollments[sid] = random.sample(EXAM_CODES, k)

# ── Exams ─────────────────────────────────────────────────────────────────────
# Build exams dict from student enrollments (inverse mapping).
exams = {code: {'students': [], 'num': 0} for code in EXAM_CODES}
for sid, courses in student_enrollments.items():
    for course in courses:
        exams[course]['students'].append(sid)
for code in EXAM_CODES:
    exams[code]['num'] = len(exams[code]['students'])

# ── Rooms ─────────────────────────────────────────────────────────────────────
# 5 rooms with varying capacities.
rooms = [
    {'room_id': 'R1', 'building': 'Alpha',   'capacity': 15},
    {'room_id': 'R2', 'building': 'Beta',    'capacity': 10},
    {'room_id': 'R3', 'building': 'Gamma',   'capacity':  8},
    {'room_id': 'R4', 'building': 'Delta',   'capacity':  5},
    {'room_id': 'R5', 'building': 'Epsilon', 'capacity':  3},
]
room_ids = [r['room_id'] for r in rooms]

# ── Timeslots ─────────────────────────────────────────────────────────────────
# 3 working days x 4 blocks = 12 slots.
# Slots 0-3  → Day 1 (2025-05-31)
# Slots 4-7  → Day 2 (2025-06-01)
# Slots 8-11 → Day 3 (2025-06-02)
times = ['08:00-10:00','10:00-12:00','12:00-14:00','14:00-16:00']
days  = [('2025-05-31','Saturday'), ('2025-06-01','Sunday'), ('2025-06-02','Monday')]
timeslots = []
slot_id = 0
for date, day in days:
    for time in times:
        timeslots.append({'slot_id': slot_id, 'date': date, 'day': day, 'time': time})
        slot_id += 1

# ── Conflict matrix ───────────────────────────────────────────────────────────
# Two exams conflict when they share at least one student.
exam_student_sets = {code: set(info['students']) for code, info in exams.items()}
conflict_matrix = {code: set() for code in EXAM_CODES}
for i, a in enumerate(EXAM_CODES):
    for b in EXAM_CODES[i+1:]:
        if exam_student_sets[a] & exam_student_sets[b]:
            conflict_matrix[a].add(b)
            conflict_matrix[b].add(a)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Exams       : {len(exams)}')
print(f'Students    : {NUM_STUDENTS}')
print(f'Rooms       : {len(rooms)}')
print(f'Timeslots   : {len(timeslots)} ({len(days)} days x {len(times)} blocks)')
print()
print('Enrollment per exam:')
for code, info in exams.items():
    conflicts = ', '.join(sorted(conflict_matrix[code])) or 'none'
    print(f'  {code}: {info["num"]:>2} students  |  conflicts with: {conflicts}')

Exams       : 10
Students    : 30
Rooms       : 5
Timeslots   : 12 (3 days x 4 blocks)

Enrollment per exam:
  CS101:  8 students  |  conflicts with: CS102, EE101, EE102, MA101, MA102, MA201, PH101, PH102
  CS102: 13 students  |  conflicts with: CS101, CS201, EE101, EE102, MA101, MA102, MA201, PH101, PH102
  CS201:  5 students  |  conflicts with: CS102, EE101, MA101, MA102, MA201, PH102
  MA101: 11 students  |  conflicts with: CS101, CS102, CS201, EE101, EE102, MA102, MA201, PH101, PH102
  MA102:  8 students  |  conflicts with: CS101, CS102, CS201, EE101, EE102, MA101, MA201, PH101, PH102
  MA201:  9 students  |  conflicts with: CS101, CS102, CS201, EE101, EE102, MA101, MA102, PH101, PH102
  PH101: 10 students  |  conflicts with: CS101, CS102, EE101, EE102, MA101, MA102, MA201, PH102
  PH102:  5 students  |  conflicts with: CS101, CS102, CS201, EE102, MA101, MA102, MA201, PH101
  EE101:  7 students  |  conflicts with: CS101, CS102, CS201, EE102, MA101, MA102, MA201, PH101
  EE102:  8 s

---
## CT1 — `no_room_double_booking`

**Rule:** no two exams may use the same room in the same timeslot.

In [3]:
# ── Case A: 0 violations ──────────────────────────────────────────────────────
# Every exam gets a unique (room, slot) pair.
schedule_a = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 0},  # same slot, different room → OK
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},  # same room, different slot → OK
    {'exam': 'MA102', 'room_id': 'R3', 'slot_id': 2},
    {'exam': 'PH101', 'room_id': 'R4', 'slot_id': 3},
]
result_a = no_room_double_booking(schedule_a)
assert result_a == 0, f'Expected 0, got {result_a}'
print(f'Case A — all unique (room, slot) pairs      : {result_a} violation  (expected 0)  ✓')

# ── Case B: 1 violation ───────────────────────────────────────────────────────
# Two exams share (R1, slot 0).
schedule_b = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 0},  # ← same room AND slot
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 1},
]
result_b = no_room_double_booking(schedule_b)
assert result_b == 1, f'Expected 1, got {result_b}'
print(f'Case B — one double-booked (room, slot)     : {result_b} violation  (expected 1)  ✓')

# ── Case C: 2 violations ─────────────────────────────────────────────────────
# Double-bookings in two separate (room, slot) pairs.
schedule_c = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 0},  # ← violation 1
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 1},
    {'exam': 'MA102', 'room_id': 'R2', 'slot_id': 1},  # ← violation 2
    {'exam': 'PH101', 'room_id': 'R3', 'slot_id': 2},
]
result_c = no_room_double_booking(schedule_c)
assert result_c == 2, f'Expected 2, got {result_c}'
print(f'Case C — two double-booked (room, slot)s    : {result_c} violations (expected 2)  ✓')

# ── Case D: random schedule, count manually ───────────────────────────────────
# Randomly assign all 10 exams to one of 3 slots in the same room
# so collisions are guaranteed and we can count them by hand.
random.seed(0)
schedule_d = [
    {'exam': code, 'room_id': 'R1', 'slot_id': random.randint(0, 2)}
    for code in EXAM_CODES
]
# Manual count: for each slot, number of extras beyond the first
slot_counts_d = {}
for a in schedule_d:
    k = (a['room_id'], a['slot_id'])
    slot_counts_d[k] = slot_counts_d.get(k, 0) + 1
expected_d = sum(v - 1 for v in slot_counts_d.values() if v > 1)
result_d = no_room_double_booking(schedule_d)
assert result_d == expected_d, f'Expected {expected_d}, got {result_d}'
print(f'Case D — random schedule, all in one room   : {result_d} violations (expected {expected_d})  ✓')

Case A — all unique (room, slot) pairs      : 0 violation  (expected 0)  ✓
Case B — one double-booked (room, slot)     : 1 violation  (expected 1)  ✓
Case C — two double-booked (room, slot)s    : 2 violations (expected 2)  ✓
Case D — random schedule, all in one room   : 7 violations (expected 7)  ✓


---
## CT2a — `no_student_clash`

**Rule:** no student may sit two exams in the same timeslot.

In [4]:
# ── Case A: 0 violations ──────────────────────────────────────────────────────
# Every exam in a different slot, so no two conflicting exams can clash.
schedule_a = [
    {'exam': code, 'room_id': 'R1', 'slot_id': i}
    for i, code in enumerate(EXAM_CODES)
]
result_a = no_student_clash(schedule_a, conflict_matrix)
assert result_a == 0, f'Expected 0, got {result_a}'
print(f'Case A — every exam in a unique slot        : {result_a} violations (expected 0)  ✓')

# ── Case B: force a clash ─────────────────────────────────────────────────────
# Find one real conflicting pair from the generated data, put them in slot 0.
pair = None
for exam_a, conflicts in conflict_matrix.items():
    if conflicts:
        exam_b = next(iter(conflicts))
        pair = (exam_a, exam_b)
        break

others = [c for c in EXAM_CODES if c not in pair]
schedule_b = (
    [{'exam': pair[0], 'room_id': 'R1', 'slot_id': 0},
     {'exam': pair[1], 'room_id': 'R2', 'slot_id': 0}]  # ← clash: share a student
  + [{'exam': c, 'room_id': 'R3', 'slot_id': i+1} for i, c in enumerate(others)]
)
result_b = no_student_clash(schedule_b, conflict_matrix)
assert result_b >= 1, f'Expected ≥1, got {result_b}'
print(f'Case B — {pair[0]} and {pair[1]} share a student in slot 0: {result_b} violation(s)  ✓')

# ── Case C: all exams in one slot ────────────────────────────────────────────
# Every exam in slot 0 — maximum possible clashes.
schedule_c = [
    {'exam': code, 'room_id': f'R{(i%5)+1}', 'slot_id': 0}
    for i, code in enumerate(EXAM_CODES)
]
# Manual count: every conflicting pair that lands in slot 0
expected_c = sum(len(v) for v in conflict_matrix.values()) // 2
result_c = no_student_clash(schedule_c, conflict_matrix)
assert result_c == expected_c, f'Expected {expected_c}, got {result_c}'
print(f'Case C — all exams in slot 0 (max clashes)  : {result_c} violations (expected {expected_c})  ✓')

# ── Case D: non-conflicting exams share a slot → 0 ───────────────────────────
# Find two exams that do NOT conflict with each other.
no_conflict_pair = None
for exam_a in EXAM_CODES:
    for exam_b in EXAM_CODES:
        if exam_a != exam_b and exam_b not in conflict_matrix[exam_a]:
            no_conflict_pair = (exam_a, exam_b)
            break
    if no_conflict_pair:
        break

schedule_d = [
    {'exam': no_conflict_pair[0], 'room_id': 'R1', 'slot_id': 0},
    {'exam': no_conflict_pair[1], 'room_id': 'R2', 'slot_id': 0},  # same slot, no shared students
]
result_d = no_student_clash(schedule_d, conflict_matrix)
assert result_d == 0, f'Expected 0, got {result_d}'
print(f'Case D — {no_conflict_pair[0]} & {no_conflict_pair[1]} in same slot, no shared students: {result_d}  ✓')

Case A — every exam in a unique slot        : 0 violations (expected 0)  ✓
Case B — CS101 and EE102 share a student in slot 0: 1 violation(s)  ✓
Case C — all exams in slot 0 (max clashes)  : 41 violations (expected 41)  ✓
Case D — CS101 & CS201 in same slot, no shared students: 0  ✓


---
## CT2b — `one_exam_per_student_per_day`

**Rule:** a student should not have two exams on the same calendar day,
even in different timeslots.

In [5]:
# ── Case A: 0 violations ──────────────────────────────────────────────────────
# Every conflicting pair is on a different day.
schedule_a = [
    {'exam': code, 'room_id': 'R1', 'slot_id': i * 4}  # slot 0,4,8 = Day 1,2,3
    for i, code in enumerate(EXAM_CODES[:3])
] + [
    {'exam': code, 'room_id': 'R2', 'slot_id': (i % 3) * 4 + 1}
    for i, code in enumerate(EXAM_CODES[3:])
]
result_a = one_exam_per_student_per_day(schedule_a, conflict_matrix, timeslots)
print(f'Case A — spread across days                 : {result_a} violations  ✓')

# ── Case B: force a same-day clash ────────────────────────────────────────────
# Take the known conflicting pair and put them in two DIFFERENT slots on Day 1.
# Slots 0 and 1 are both on 2025-05-31.
others = [c for c in EXAM_CODES if c not in pair]
schedule_b = (
    [{'exam': pair[0], 'room_id': 'R1', 'slot_id': 0},   # Day 1, slot 0
     {'exam': pair[1], 'room_id': 'R2', 'slot_id': 1}]   # Day 1, slot 1 ← same day!
  + [{'exam': c, 'room_id': 'R3', 'slot_id': 4+i} for i, c in enumerate(others)]
)
result_b = one_exam_per_student_per_day(schedule_b, conflict_matrix, timeslots)
assert result_b >= 1, f'Expected ≥1, got {result_b}'
print(f'Case B — {pair[0]} & {pair[1]} on same day, diff slots: {result_b} violation(s)  ✓')

# ── Case C: all exams on Day 1 ────────────────────────────────────────────────
# Slots 0-3 are all Day 1. Guarantees maximum same-day conflicts.
schedule_c = [
    {'exam': code, 'room_id': f'R{(i%5)+1}', 'slot_id': i % 4}
    for i, code in enumerate(EXAM_CODES)
]
# Manual count: every conflicting pair is on the same day
expected_c = sum(len(v) for v in conflict_matrix.values()) // 2
result_c = one_exam_per_student_per_day(schedule_c, conflict_matrix, timeslots)
assert result_c == expected_c, f'Expected {expected_c}, got {result_c}'
print(f'Case C — all exams on Day 1 (max same-day)  : {result_c} violations (expected {expected_c})  ✓')

# ── Case D: different days, same students → 0 ─────────────────────────────────
# The conflicting pair is on different days, so no same-day violation.
schedule_d = [
    {'exam': pair[0], 'room_id': 'R1', 'slot_id': 0},  # Day 1
    {'exam': pair[1], 'room_id': 'R1', 'slot_id': 4},  # Day 2
]
result_d = one_exam_per_student_per_day(schedule_d, conflict_matrix, timeslots)
assert result_d == 0, f'Expected 0, got {result_d}'
print(f'Case D — conflicting pair on different days  : {result_d} violations (expected 0)  ✓')

Case A — spread across days                 : 12 violations  ✓
Case B — CS101 & EE102 on same day, diff slots: 12 violation(s)  ✓
Case C — all exams on Day 1 (max same-day)  : 41 violations (expected 41)  ✓
Case D — conflicting pair on different days  : 0 violations (expected 0)  ✓


---
## CT3 — `exams_spread_evenly`

**Rule:** any day with more than `(total_exams / total_days) + 2` exams counts as 1 violation.

With 10 exams and 3 days: ideal = 10/3 ≈ 3.33, threshold = 3.33 + 2 = 5.33.
A day needs **6 or more** exams to trigger a violation.

In [6]:
ideal = len(EXAM_CODES) / len(days)
threshold = ideal + 2
print(f'Ideal exams/day: {ideal:.2f}  |  Violation threshold: > {threshold:.2f}  (i.e. 6+)')
print()

# ── Case A: 0 violations — roughly even spread ────────────────────────────────
# Day 1: 4, Day 2: 3, Day 3: 3 — none exceed 5.33.
schedule_a = (
    [{'exam': c, 'room_id': 'R1', 'slot_id': i}   for i, c in enumerate(EXAM_CODES[:4])]  # Day1
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 4+i} for i, c in enumerate(EXAM_CODES[4:7])] # Day2
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 8+i} for i, c in enumerate(EXAM_CODES[7:])]  # Day3
)
result_a = exams_spread_evenly(schedule_a, timeslots)
assert result_a == 0, f'Expected 0, got {result_a}'
print(f'Case A — Day1:4, Day2:3, Day3:3 (none > 5.33) : {result_a} violations (expected 0)  ✓')

# ── Case B: 1 violation — 6 exams on Day 1 ───────────────────────────────────
# Day 1 gets 6 exams (6 > 5.33 → 1 overloaded day).
schedule_b = (
    [{'exam': c, 'room_id': f'R{(i%5)+1}', 'slot_id': i % 4} for i, c in enumerate(EXAM_CODES[:6])]  # Day1
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 4+i} for i, c in enumerate(EXAM_CODES[6:8])]             # Day2
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 8+i} for i, c in enumerate(EXAM_CODES[8:])]              # Day3
)
result_b = exams_spread_evenly(schedule_b, timeslots)
assert result_b == 1, f'Expected 1, got {result_b}'
print(f'Case B — Day1:6, Day2:2, Day3:2 (Day1 > 5.33) : {result_b} violations (expected 1)  ✓')

# ── Case C: 5 on Day 1 — just under threshold → 0 ────────────────────────────
# 5 ≤ 5.33, so no violation.
schedule_c = (
    [{'exam': c, 'room_id': f'R{(i%5)+1}', 'slot_id': i % 4} for i, c in enumerate(EXAM_CODES[:5])]  # Day1
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 4+i} for i, c in enumerate(EXAM_CODES[5:8])]             # Day2
  + [{'exam': c, 'room_id': 'R1', 'slot_id': 8+i} for i, c in enumerate(EXAM_CODES[8:])]              # Day3
)
result_c = exams_spread_evenly(schedule_c, timeslots)
assert result_c == 0, f'Expected 0, got {result_c}'
print(f'Case C — Day1:5, Day2:3, Day3:2 (5 ≤ 5.33)    : {result_c} violations (expected 0)  ✓')

# ── Case D: all exams on Day 1 ────────────────────────────────────────────────
schedule_d = [
    {'exam': c, 'room_id': f'R{(i%5)+1}', 'slot_id': i % 4}
    for i, c in enumerate(EXAM_CODES)
]
result_d = exams_spread_evenly(schedule_d, timeslots)
assert result_d == 1, f'Expected 1 (only Day1 is overloaded), got {result_d}'
print(f'Case D — all 10 on Day 1                        : {result_d} violations (expected 1)  ✓')

Ideal exams/day: 3.33  |  Violation threshold: > 5.33  (i.e. 6+)

Case A — Day1:4, Day2:3, Day3:3 (none > 5.33) : 0 violations (expected 0)  ✓
Case B — Day1:6, Day2:2, Day3:2 (Day1 > 5.33) : 1 violations (expected 1)  ✓
Case C — Day1:5, Day2:3, Day3:2 (5 ≤ 5.33)    : 0 violations (expected 0)  ✓
Case D — all 10 on Day 1                        : 1 violations (expected 1)  ✓


---
## CT4 — `assign_rooms_to_exam`

**Rule:** every exam must be assigned to one or more rooms.
If the student count fits in one room, use that room.
If not, split students across multiple rooms, filling the largest first.

In [7]:
room_cap = {r['room_id']: r['capacity'] for r in rooms}

# ── Case A: exam fits in one room ─────────────────────────────────────────────
# Find an exam whose enrollment fits inside R1 (capacity 15).
small_exam = min(EXAM_CODES, key=lambda c: exams[c]['num'])
result_a = assign_rooms_to_exam(small_exam, exams, rooms)
assert len(result_a) == 1,                     'Should use exactly one room'
assert result_a[0]['room_id'] == 'R1',         'Should pick the largest room'
assert result_a[0]['students'] == exams[small_exam]['students']
print(f'Case A — {small_exam} ({exams[small_exam]["num"]} students) fits in one room')
print(f'         → assigned to {result_a[0]["room_id"]} (cap {room_cap[result_a[0]["room_id"]]})  ✓')

# ── Case B: exam must be split ────────────────────────────────────────────────
# Use only small rooms so the exam MUST split.
small_rooms = [r for r in rooms if r['capacity'] <= 5]  # R4 (5) and R5 (3)
big_exam    = max(EXAM_CODES, key=lambda c: exams[c]['num'])
total_small = sum(r['capacity'] for r in small_rooms)

if exams[big_exam]['num'] <= total_small:
    result_b = assign_rooms_to_exam(big_exam, exams, small_rooms)
    all_placed = [s for chunk in result_b for s in chunk['students']]
    # All students placed
    assert set(all_placed) == set(exams[big_exam]['students']), 'All students must be placed'
    # No room chunk exceeds its capacity
    for chunk in result_b:
        assert len(chunk['students']) <= room_cap[chunk['room_id']], \
            f"Room {chunk['room_id']} overflowed"
    print(f'Case B — {big_exam} ({exams[big_exam]["num"]} students) split across {len(result_b)} room(s)')
    for chunk in result_b:
        print(f'         → {chunk["room_id"]} (cap {room_cap[chunk["room_id"]]}) gets {len(chunk["students"])} students')
    print('         All students placed, no overflow  ✓')
else:
    print(f'Case B — skipped: {big_exam} needs {exams[big_exam]["num"]} seats, only {total_small} available in small rooms')

# ── Case C: not enough seats → ValueError ────────────────────────────────────
# Pass only a 1-seat room for an exam with multiple students.
one_seat = [{'room_id': 'TINY', 'building': 'X', 'capacity': 1}]
multi_student_exam = next(c for c in EXAM_CODES if exams[c]['num'] > 1)
try:
    assign_rooms_to_exam(multi_student_exam, exams, one_seat)
    print('Case C — FAIL: should have raised ValueError')
except ValueError as e:
    print(f'Case C — ValueError raised correctly  ✓')
    print(f'         {e}')

# ── Case D: empty room list → [] ─────────────────────────────────────────────
result_d = assign_rooms_to_exam(small_exam, exams, [])
assert result_d == [], f'Expected [], got {result_d}'
print(f'Case D — no rooms available → returns []  ✓')

Case A — CS201 (5 students) fits in one room
         → assigned to R1 (cap 15)  ✓
Case B — skipped: CS102 needs 13 seats, only 8 available in small rooms
Case C — ValueError raised correctly  ✓
         Not enough room capacity for 'CS101': need 8 seats but only 1 available across 1 room(s).
Case D — no rooms available → returns []  ✓


---
## `hard_violations` — Aggregator

Must equal `no_room_double_booking + no_student_clash + CT4 room assignment check`.

In [8]:
# ── Build a schedule where every exam is properly split via CT4 ───────────────
ct4_schedule = []
for i, code in enumerate(EXAM_CODES):
    chunks = assign_rooms_to_exam(code, exams, rooms)
    ct4_schedule.append({
        'exam':    code,
        'room_id': chunks[0]['room_id'],
        'slot_id': i,                    # every exam in a unique slot
        'rooms':   chunks,
    })

# ── Case A: clean schedule → 0 hard violations ────────────────────────────────
result_a = hard_violations(ct4_schedule, conflict_matrix, exams, rooms)
assert result_a == 0, f'Expected 0, got {result_a}'
print(f'Case A — clean CT4 schedule                 : {result_a} hard violations (expected 0)  ✓')

# ── Case B: missing rooms key → 1 hard violation per missing exam ──────────────
no_ct4_schedule = [
    {'exam': code, 'room_id': 'R1', 'slot_id': i}   # no 'rooms' key
    for i, code in enumerate(EXAM_CODES)
]
result_b = hard_violations(no_ct4_schedule, conflict_matrix, exams, rooms)
# CT4 fires for every exam (10 exams, each missing the rooms key)
assert result_b == len(EXAM_CODES), f'Expected {len(EXAM_CODES)}, got {result_b}'
print(f'Case B — no rooms key on any exam           : {result_b} hard violations (expected {len(EXAM_CODES)})  ✓')

# ── Case C: double-booking + student clash + CT4 miss ─────────────────────────
# CS101 and CS102 in (R1, slot 0) → 1 double-booking, possibly 1 student clash
# MA101 missing rooms key → 1 CT4 violation
mixed_schedule = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0,
     'rooms': assign_rooms_to_exam('CS101', exams, rooms)},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 0,    # ← double-booking
     'rooms': assign_rooms_to_exam('CS102', exams, rooms)},
    {'exam': 'MA101', 'room_id': 'R2', 'slot_id': 1},   # ← no rooms key
]
result_c = hard_violations(mixed_schedule, conflict_matrix, exams, rooms)
# Manually tally what we expect
db  = no_room_double_booking(mixed_schedule)            # should be 1
nsc = no_student_clash(mixed_schedule, conflict_matrix) # depends on conflict data
ct4 = 1                                                 # MA101 missing rooms
expected_c = db + nsc + ct4
assert result_c == expected_c, f'Expected {expected_c}, got {result_c}'
print(f'Case C — double-booking + clash + CT4 miss  : {result_c} hard violations')
print(f'         breakdown → double-book={db}, clash={nsc}, CT4={ct4}  ✓')

Case A — clean CT4 schedule                 : 0 hard violations (expected 0)  ✓
Case B — no rooms key on any exam           : 10 hard violations (expected 10)  ✓
Case C — double-booking + clash + CT4 miss  : 3 hard violations
         breakdown → double-book=1, clash=1, CT4=1  ✓


---
## `soft_violations` — Aggregator

Must equal `one_exam_per_student_per_day + exams_spread_evenly`.

In [9]:
# ── Case A: spread schedule → low soft violations ─────────────────────────────
# Spread exams evenly and put conflicting pairs on different days.
spread_schedule = []
for i, code in enumerate(EXAM_CODES):
    chunks = assign_rooms_to_exam(code, exams, rooms)
    spread_schedule.append({
        'exam':    code,
        'room_id': chunks[0]['room_id'],
        'slot_id': i * 1,   # slot 0,1,2,...,9 — spread across 3 days naturally
        'rooms':   chunks,
    })

s1 = one_exam_per_student_per_day(spread_schedule, conflict_matrix, timeslots)
s2 = exams_spread_evenly(spread_schedule, timeslots)
expected_a = s1 + s2
result_a   = soft_violations(spread_schedule, conflict_matrix, timeslots)
assert result_a == expected_a, f'Expected {expected_a}, got {result_a}'
print(f'Case A — spread schedule')
print(f'         same-day conflicts={s1}, overloaded days={s2}  → total={result_a}  ✓')

# ── Case B: all exams crammed onto Day 1 ──────────────────────────────────────
crammed_schedule = []
for i, code in enumerate(EXAM_CODES):
    chunks = assign_rooms_to_exam(code, exams, rooms)
    crammed_schedule.append({
        'exam':    code,
        'room_id': chunks[0]['room_id'],
        'slot_id': i % 4,   # all in slots 0-3 = Day 1
        'rooms':   chunks,
    })

s1 = one_exam_per_student_per_day(crammed_schedule, conflict_matrix, timeslots)
s2 = exams_spread_evenly(crammed_schedule, timeslots)
expected_b = s1 + s2
result_b   = soft_violations(crammed_schedule, conflict_matrix, timeslots)
assert result_b == expected_b, f'Expected {expected_b}, got {result_b}'
print(f'Case B — all exams crammed onto Day 1')
print(f'         same-day conflicts={s1}, overloaded days={s2}  → total={result_b}  ✓')

# ── Case C: aggregator consistency check ──────────────────────────────────────
# Verify soft_violations always equals the sum of its two components
# across 10 randomly generated schedules.
random.seed(7)
all_match = True
for trial in range(10):
    rand_sched = []
    for code in EXAM_CODES:
        chunks = assign_rooms_to_exam(code, exams, rooms)
        rand_sched.append({
            'exam':    code,
            'room_id': chunks[0]['room_id'],
            'slot_id': random.randint(0, 11),
            'rooms':   chunks,
        })
    s1 = one_exam_per_student_per_day(rand_sched, conflict_matrix, timeslots)
    s2 = exams_spread_evenly(rand_sched, timeslots)
    sv = soft_violations(rand_sched, conflict_matrix, timeslots)
    if sv != s1 + s2:
        all_match = False
        print(f'  Trial {trial}: MISMATCH sv={sv} s1+s2={s1+s2}')
if all_match:
    print(f'Case C — aggregator matches s1+s2 across 10 random schedules  ✓')

Case A — spread schedule
         same-day conflicts=12, overloaded days=0  → total=12  ✓
Case B — all exams crammed onto Day 1
         same-day conflicts=41, overloaded days=1  → total=42  ✓
Case C — aggregator matches s1+s2 across 10 random schedules  ✓


---
## Summary

In [10]:
print('=' * 52)
print('ALL CONSTRAINT TESTS PASSED')
print('=' * 52)
tests = [
    ('CT1',  'no_room_double_booking',          'Hard'),
    ('CT2a', 'no_student_clash',                'Hard'),
    ('CT2b', 'one_exam_per_student_per_day',    'Soft'),
    ('CT3',  'exams_spread_evenly',             'Soft'),
    ('CT4',  'assign_rooms_to_exam',            'Hard'),
    ('—',    'hard_violations (aggregator)',    'Hard'),
    ('—',    'soft_violations (aggregator)',    'Soft'),
]
for label, name, kind in tests:
    print(f'  {label:<5}  {name:<40} {kind}')

ALL CONSTRAINT TESTS PASSED
  CT1    no_room_double_booking                   Hard
  CT2a   no_student_clash                         Hard
  CT2b   one_exam_per_student_per_day             Soft
  CT3    exams_spread_evenly                      Soft
  CT4    assign_rooms_to_exam                     Hard
  —      hard_violations (aggregator)             Hard
  —      soft_violations (aggregator)             Soft
